<a href="https://colab.research.google.com/github/Alister44/Cafe_Sales_Analysis/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
FILE_IN = 'https://raw.githubusercontent.com/Alister44/Cafe_Sales_Analysis/refs/heads/main/data/dirty_cafe_sales.csv'

In [3]:
df_origin = pd.read_csv(FILE_IN)
df = df_origin.copy()

In [4]:
# Drop the technical index column if it exists
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [5]:
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [7]:
def info_types_missVals(data_frame):
    """Show the data type and the number and share of missing values for each column."""
    return pd.DataFrame({
        "dtype": data_frame.dtypes,
        "missing count": data_frame.isnull().sum(),
        "missing (%)": (data_frame.isna().mean() * 100).round(2)
    }).sort_values("missing (%)", ascending=False)

In [8]:
info_types_missVals(df)

,dtype,missing count,missing (%)
Location,object,3265,32.65
Payment Method,object,2579,25.79
Item,object,333,3.33
Price Per Unit,object,179,1.79
Total Spent,object,173,1.73
Transaction Date,object,159,1.59
Quantity,object,138,1.38
Transaction ID,object,0,0.00


In [9]:
#Duplicate check
df.duplicated().sum()

np.int64(0)

In [10]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [11]:
display(df['Item'].value_counts(dropna=False))

,count
Item,
Juice,1171
Coffee,1165
Salad,1148
Cake,1139
Sandwich,1131
Smoothie,1096
Cookie,1092
Tea,1089
UNKNOWN,344


In [12]:
display(df['Payment Method'].value_counts(dropna=False))

,count
Payment Method,
NaN,2579
Digital Wallet,2291
Credit Card,2273
Cash,2258
ERROR,306
UNKNOWN,293


In [13]:
display(df['Location'].value_counts(dropna=False))

,count
Location,
NaN,3265
Takeaway,3022
In-store,3017
ERROR,358
UNKNOWN,338


In [14]:
df.isin(['ERROR','UNKNOWN']).sum()

,0
Transaction ID,0
Item,636
Quantity,341
Price Per Unit,354
Total Spent,329
Payment Method,599
Location,696
Transaction Date,301


In [18]:
import numpy as np

In [19]:
df.replace(['ERROR', 'UNKNOWN', 'error', 'unknown', '', ' '], np.nan, inplace=True)

In [20]:
df.isin(['ERROR','UNKNOWN']).sum()

,0
transaction_id,0
item,0
quantity,0
price_per_unit,0
total_spent,0
payment_method,0
location,0
transaction_date,0


In [15]:
# put our column name into snake_case
df.columns = (df.columns
    .str.strip() #delete spaces in the beginnings and ends but not in the middle
    .str.lower() # makes every letter lower
    .str.replace(" ", "_"))

In [16]:
df.columns

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='object')

In [21]:
info_types_missVals(df)

,dtype,missing count,missing (%)
location,object,3961,39.61
payment_method,object,3178,31.78
item,object,969,9.69
price_per_unit,object,533,5.33
total_spent,object,502,5.02
quantity,object,479,4.79
transaction_date,object,460,4.60
transaction_id,object,0,0.00


In [22]:
df.head()

,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,NaN,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [35]:
# Convert data types
cat_cols = ['item','payment_method','location']
df[cat_cols] = df[cat_cols].astype('category')

df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype('Int64')
df['price_per_unit'] = pd.to_numeric(df['price_per_unit'], errors='coerce')
df['total_spent'] = pd.to_numeric(df['total_spent'], errors='coerce')

df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

df.dtypes

,0
transaction_id,object
item,category
quantity,Int64
price_per_unit,float64
total_spent,float64
payment_method,category
location,category
transaction_date,datetime64[ns]


In [37]:
info_types_missVals(df)

,dtype,missing count,missing (%)
location,category,3961,39.61
payment_method,category,3178,31.78
item,category,969,9.69
price_per_unit,float64,533,5.33
total_spent,float64,502,5.02
quantity,Int64,479,4.79
transaction_date,datetime64[ns],460,4.60
transaction_id,object,0,0.00


In [38]:
# Fill missing values (total_spent = price_per_unit * quantity)
# Fill quantity
mask_q = df.quantity.isna() & df.price_per_unit.notna() & df.total_spent.notna()
df.loc[mask_q,'quantity'] = df.total_spent / df.price_per_unit


,transaction_id,item,quantity,price_per_unit,total_spent,payment_method,location,transaction_date
20,TXN_3522028,Smoothie,<NA>,4.0,20.0,Cash,In-store,2023-04-04
55,TXN_5522862,Cookie,<NA>,1.0,2.0,Credit Card,Takeaway,2023-03-19
57,TXN_2080895,Cake,<NA>,3.0,3.0,Digital Wallet,In-store,2023-04-19
66,TXN_8501819,Juice,<NA>,3.0,6.0,Cash,NaN,2023-03-30
117,TXN_2148617,Juice,<NA>,3.0,9.0,Digital Wallet,NaN,2023-01-10
...,...,...,...,...,...,...,...,...
9932,TXN_8502079,Tea,<NA>,1.5,3.0,Cash,NaN,2023-04-20
9935,TXN_9778251,Tea,<NA>,1.5,6.0,NaN,Takeaway,2023-11-09
9944,TXN_7495283,Cake,<NA>,3.0,15.0,Credit Card,Takeaway,2023-04-14
9957,TXN_6487003,Coffee,<NA>,2.0,8.0,Credit Card,Takeaway,2023-11-15
